In [6]:
from pathlib import Path

print(Path("../data/raw/train.json").resolve())
print(Path("../data/raw/train.json").exists())

print(Path("../data/raw/val.json").resolve())
print(Path("../data/raw/val.json").exists())

print(Path("../data/raw/test.json").resolve())
print(Path("../data/raw/test.json").exists())

/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/train.json
True
/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/val.json
True
/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/test.json
True


In [8]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "../data/raw/train.json",
        "validation": "../data/raw/val.json",
        "test": "../data/raw/test.json"
    }
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 10000
    })
})


In [9]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "../data/raw/train.json",
        "validation": "../data/raw/val.json",
        "test": "../data/raw/test.json"
    }
)

print(dataset)
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print("Test size:", len(dataset["test"]))
print("Columns:", dataset["train"].column_names)
print("First train example:", dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 10000
    })
})
Train size: 20000
Validation size: 10000
Test size: 10000
Columns: ['dialogue', 'summary', 'summary_de', 'summary_zh']
First train example: {'dialogue': "MADELEINE BRAND, host: I'm Madeleine Brand. Chinese Premier Wen Jiabao is in Sichuan province today. He's inspecting the so called quake lake, that's a lake formed by landslides after last month's 7.9 earthquake.\r\nALEX CHADWICK, host: Authorities have evacuated one quarter of a million people in the flood path of that lake. For some it's their second flight to safety, and still many are anxious to get back home. Jamila Trindle reports.\r\nJAMILA TRIN

In [10]:
!pip install jieba


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
import jieba

def add_length_features(example):
    example["dialogue_char_len"] = len(example["dialogue"])
    example["dialogue_word_len"] = len(example["dialogue"].split())

    example["summary_char_len"] = len(example["summary"])
    example["summary_word_len"] = len(example["summary"].split())

    example["summary_zh_char_len"] = len(example["summary_zh"])
    example["summary_zh_word_len"] = len(jieba.lcut(example["summary_zh"]))

    return example

dataset_len = dataset.map(add_length_features)

/Applications/anaconda3/envs/nlp/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/wv/6g0vndhj1918ng7mb558jqbh0000gn/T/jieba.cache
Loading model cost 0.256 seconds.
Prefix dict has been built successfully.


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [12]:
for split in ["train", "validation", "test"]:
    print(f"[{split}]")
    print("dialogue avg char:", round(sum(dataset_len[split]["dialogue_char_len"]) / len(dataset_len[split]), 2))
    print("dialogue avg word:", round(sum(dataset_len[split]["dialogue_word_len"]) / len(dataset_len[split]), 2))
    print("summary avg char:", round(sum(dataset_len[split]["summary_char_len"]) / len(dataset_len[split]), 2))
    print("summary avg word:", round(sum(dataset_len[split]["summary_word_len"]) / len(dataset_len[split]), 2))
    print("summary_zh avg char:", round(sum(dataset_len[split]["summary_zh_char_len"]) / len(dataset_len[split]), 2))
    print("summary_zh avg word:", round(sum(dataset_len[split]["summary_zh_word_len"]) / len(dataset_len[split]), 2))
    print()

[train]
dialogue avg char: 9263.87
dialogue avg word: 1623.5
summary avg char: 92.72
summary avg word: 14.45
summary_zh avg char: 30.9
summary_zh avg word: 16.39

[validation]
dialogue avg char: 9047.18
dialogue avg word: 1586.26
summary avg char: 91.04
summary avg word: 14.18
summary_zh avg char: 30.61
summary_zh avg word: 16.18

[test]
dialogue avg char: 9244.17
dialogue avg word: 1621.67
summary avg char: 91.9
summary avg word: 14.34
summary_zh avg char: 30.98
summary_zh avg word: 16.41



In [13]:
def overall_avg(dataset, field, mode="char"):
    if mode == "char":
        lengths = [len(x[field]) for split in ["train", "validation", "test"] for x in dataset[split]]
    elif mode == "word":
        if field in ["dialogue", "summary"]:
            lengths = [len(x[field].split()) for split in ["train", "validation", "test"] for x in dataset[split]]
        elif field == "summary_zh":
            import jieba
            lengths = [len(jieba.lcut(x[field])) for split in ["train", "validation", "test"] for x in dataset[split]]
    return sum(lengths) / len(lengths)

print("dialogue overall avg char:", round(overall_avg(dataset, "dialogue", "char"), 2))
print("dialogue overall avg word:", round(overall_avg(dataset, "dialogue", "word"), 2))

print("summary overall avg char:", round(overall_avg(dataset, "summary", "char"), 2))
print("summary overall avg word:", round(overall_avg(dataset, "summary", "word"), 2))

print("summary_zh overall avg char:", round(overall_avg(dataset, "summary_zh", "char"), 2))
print("summary_zh overall avg word:", round(overall_avg(dataset, "summary_zh", "word"), 2))

dialogue overall avg char: 9204.77
dialogue overall avg word: 1613.73
summary overall avg char: 92.1
summary overall avg word: 14.36
summary_zh overall avg char: 30.85
summary_zh overall avg word: 16.34


In [14]:
import pandas as pd

# Compute compression ratios for each split
results = []

for split in ["train", "validation", "test"]:
    token_ratios = []
    char_ratios = []

    for example in dataset[split]:
        dialogue = example["dialogue"]
        summary = example["summary"]

        dialogue_token_count = len(dialogue.split())
        summary_token_count = len(summary.split())

        dialogue_char_count = len(dialogue)
        summary_char_count = len(summary)

        if summary_token_count > 0:
            token_ratios.append(dialogue_token_count / summary_token_count)

        if summary_char_count > 0:
            char_ratios.append(dialogue_char_count / summary_char_count)

    results.append({
        "split": split,
        "avg_token_compression_ratio": sum(token_ratios) / len(token_ratios),
        "avg_char_compression_ratio": sum(char_ratios) / len(char_ratios),
    })

df_compression = pd.DataFrame(results).round(2)
print(df_compression)

        split  avg_token_compression_ratio  avg_char_compression_ratio
0       train                       166.84                      147.31
1  validation                       164.28                      145.16
2        test                       167.35                      147.68


In [15]:
# Compute overall compression ratios across all splits
all_token_ratios = []
all_char_ratios = []

for split in ["train", "validation", "test"]:
    for example in dataset[split]:
        dialogue = example["dialogue"]
        summary = example["summary"]

        dialogue_token_count = len(dialogue.split())
        summary_token_count = len(summary.split())

        dialogue_char_count = len(dialogue)
        summary_char_count = len(summary)

        if summary_token_count > 0:
            all_token_ratios.append(dialogue_token_count / summary_token_count)

        if summary_char_count > 0:
            all_char_ratios.append(dialogue_char_count / summary_char_count)

print("Overall avg token compression ratio:", round(sum(all_token_ratios) / len(all_token_ratios), 2))
print("Overall avg char compression ratio:", round(sum(all_char_ratios) / len(all_char_ratios), 2))

Overall avg token compression ratio: 166.33
Overall avg char compression ratio: 146.87


In [16]:
def add_compression_features(example):
    dialogue = example["dialogue"]
    summary = example["summary"]

    dialogue_token_count = len(dialogue.split())
    summary_token_count = len(summary.split())

    dialogue_char_count = len(dialogue)
    summary_char_count = len(summary)

    example["dialogue_token_count"] = dialogue_token_count
    example["summary_token_count"] = summary_token_count
    example["dialogue_char_count"] = dialogue_char_count
    example["summary_char_count"] = summary_char_count

    example["token_compression_ratio"] = (
        dialogue_token_count / summary_token_count if summary_token_count > 0 else None
    )
    example["char_compression_ratio"] = (
        dialogue_char_count / summary_char_count if summary_char_count > 0 else None
    )

    return example

dataset_with_compression = dataset.map(add_compression_features)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [17]:
for split in ["train", "validation", "test"]:
    token_ratios = [x for x in dataset_with_compression[split]["token_compression_ratio"] if x is not None]
    char_ratios = [x for x in dataset_with_compression[split]["char_compression_ratio"] if x is not None]

    print(f"[{split}]")
    print("avg token compression ratio:", round(sum(token_ratios) / len(token_ratios), 2))
    print("avg char compression ratio:", round(sum(char_ratios) / len(char_ratios), 2))
    print()

[train]
avg token compression ratio: 166.84
avg char compression ratio: 147.31

[validation]
avg token compression ratio: 164.28
avg char compression ratio: 145.16

[test]
avg token compression ratio: 167.35
avg char compression ratio: 147.68



In [18]:
import pandas as pd
import jieba

# Define tokenizers for each field
def tokenize_english(text):
    return text.split()

def tokenize_chinese(text):
    return jieba.lcut(text)

tokenizer_map = {
    "dialogue": tokenize_english,
    "summary": tokenize_english,
    "summary_zh": tokenize_chinese,
}

fields = ["dialogue", "summary", "summary_zh"]
splits = ["train", "validation", "test"]

results = []

# Compute min, max, and average token counts for each split and field
for split in splits:
    for field in fields:
        token_counts = [
            len(tokenizer_map[field](example[field]))
            for example in dataset[split]
        ]

        results.append({
            "split": split,
            "field": field,
            "min_token_count": min(token_counts),
            "max_token_count": max(token_counts),
            "avg_token_count": round(sum(token_counts) / len(token_counts), 2),
        })

df_token_stats = pd.DataFrame(results)
print(df_token_stats)

        split       field  min_token_count  max_token_count  avg_token_count
0       train    dialogue               41            32904          1623.50
1       train     summary                5              186            14.45
2       train  summary_zh                1              228            16.39
3  validation    dialogue               35            17642          1586.26
4  validation     summary                5              185            14.18
5  validation  summary_zh                2              230            16.18
6        test    dialogue               17            19943          1621.67
7        test     summary                5              185            14.34
8        test  summary_zh                1              220            16.41


In [19]:
overall_results = []

# Compute min, max, and average token counts across all splits
for field in fields:
    token_counts = []

    for split in splits:
        token_counts.extend(
            len(tokenizer_map[field](example[field]))
            for example in dataset[split]
        )

    overall_results.append({
        "split": "overall",
        "field": field,
        "min_token_count": min(token_counts),
        "max_token_count": max(token_counts),
        "avg_token_count": round(sum(token_counts) / len(token_counts), 2),
    })

df_overall_token_stats = pd.DataFrame(overall_results)
print(df_overall_token_stats)

     split       field  min_token_count  max_token_count  avg_token_count
0  overall    dialogue               17            32904          1613.73
1  overall     summary                5              186            14.36
2  overall  summary_zh                1              230            16.34


In [20]:
for split in splits:
    print(f"\n[{split}]")
    for field in fields:
        token_counts = [
            len(tokenizer_map[field](example[field]))
            for example in dataset[split]
        ]

        print(
            f"{field}: "
            f"min={min(token_counts)}, "
            f"max={max(token_counts)}, "
            f"avg={round(sum(token_counts) / len(token_counts), 2)}"
        )


[train]
dialogue: min=41, max=32904, avg=1623.5
summary: min=5, max=186, avg=14.45
summary_zh: min=1, max=228, avg=16.39

[validation]
dialogue: min=35, max=17642, avg=1586.26
summary: min=5, max=185, avg=14.18
summary_zh: min=2, max=230, avg=16.18

[test]
dialogue: min=17, max=19943, avg=1621.67
summary: min=5, max=185, avg=14.34
summary_zh: min=1, max=220, avg=16.41


In [21]:
import jieba

# Find train examples where summary_zh has exactly 1 token
for i, example in enumerate(dataset["train"]):
    tokens = jieba.lcut(example["summary_zh"])
    if len(tokens) == 1:
        print(f"Index: {i}")
        print("summary_zh:", repr(example["summary_zh"]))
        print("tokens:", tokens)
        print("char length:", len(example["summary_zh"]))
        print("-" * 50)

Index: 10010
summary_zh: '三代同堂'
tokens: ['三代同堂']
char length: 4
--------------------------------------------------
Index: 11807
summary_zh: '贫富差距'
tokens: ['贫富差距']
char length: 4
--------------------------------------------------


In [22]:
# Missing value check
import pandas as pd

fields = ["dialogue", "summary", "summary_zh"]
splits = ["train", "validation", "test"]

results = []

# Check missing-like values for each split and field
for split in splits:
    for field in fields:
        none_count = 0
        empty_count = 0
        whitespace_count = 0

        for example in dataset[split]:
            value = example[field]

            if value is None:
                none_count += 1
            elif isinstance(value, str):
                if value == "":
                    empty_count += 1
                elif value.strip() == "":
                    whitespace_count += 1

        results.append({
            "split": split,
            "field": field,
            "none_count": none_count,
            "empty_string_count": empty_count,
            "whitespace_only_count": whitespace_count,
        })

df_missing = pd.DataFrame(results)
print(df_missing)

        split       field  none_count  empty_string_count  \
0       train    dialogue           0                   0   
1       train     summary           0                   0   
2       train  summary_zh           0                   0   
3  validation    dialogue           0                   0   
4  validation     summary           0                   0   
5  validation  summary_zh           0                   0   
6        test    dialogue           0                   0   
7        test     summary           0                   0   
8        test  summary_zh           0                   0   

   whitespace_only_count  
0                      0  
1                      0  
2                      0  
3                      0  
4                      0  
5                      0  
6                      0  
7                      0  
8                      0  


In [24]:
from collections import Counter
import pandas as pd

splits = ["train", "validation", "test"]

results = []

# Check duplicates within each split
for split in splits:
    dialogue_list = [example["dialogue"] for example in dataset[split]]
    full_sample_list = [
        (example["dialogue"], example["summary"], example["summary_zh"])
        for example in dataset[split]
    ]

    dialogue_counter = Counter(dialogue_list)
    full_sample_counter = Counter(full_sample_list)

    duplicate_dialogues = sum(1 for count in dialogue_counter.values() if count > 1)
    duplicate_full_samples = sum(1 for count in full_sample_counter.values() if count > 1)

    duplicated_dialogue_rows = sum(count - 1 for count in dialogue_counter.values() if count > 1)
    duplicated_full_sample_rows = sum(count - 1 for count in full_sample_counter.values() if count > 1)

    results.append({
        "split": split,
        "duplicate_dialogue_types": duplicate_dialogues,
        "duplicate_dialogue_rows": duplicated_dialogue_rows,
        "duplicate_full_sample_types": duplicate_full_samples,
        "duplicate_full_sample_rows": duplicated_full_sample_rows,
    })

df_duplicates_within = pd.DataFrame(results)
print(df_duplicates_within)

        split  duplicate_dialogue_types  duplicate_dialogue_rows  \
0       train                         4                        4   
1  validation                         2                        2   
2        test                         1                        1   

   duplicate_full_sample_types  duplicate_full_sample_rows  
0                            4                           4  
1                            1                           1  
2                            0                           0  


In [27]:
fields = ["dialogue", "summary", "summary_zh"]
splits = ["train", "validation", "test"]

# Print examples with missing-like values
for split in splits:
    print(f"\n[{split}]")
    for i, example in enumerate(dataset[split]):
        for field in fields:
            value = example[field]
            if value is None or (isinstance(value, str) and value.strip() == ""):
                print(f"Index: {i}, Field: {field}, Value: {repr(value)}")


[train]

[validation]

[test]


In [25]:
from collections import Counter

# Print duplicated dialogues within each split
for split in ["train", "validation", "test"]:
    print(f"\n[{split}] duplicated dialogues")
    dialogue_list = [example["dialogue"] for example in dataset[split]]
    dialogue_counter = Counter(dialogue_list)

    duplicated_items = [(text, count) for text, count in dialogue_counter.items() if count > 1]

    for idx, (text, count) in enumerate(duplicated_items[:5]):
        print(f"\nDuplicate #{idx + 1} | Count: {count}")
        print(repr(text[:500]))


[train] duplicated dialogues

Duplicate #1 | Count: 2
'ROTH: Free speech lives at the U.N., but please, if you\'re going to say something like that, bring a picture or give out a phone number. Well there is a gorgeous woman whose name is not Irina (ph), but Afsane, and she\'s here in the studio with me. Afsane Bassir Pour of the French newspaper "Le Monde," and over at Stringfellows (ph) -- I mean at the U.N. is James Bone of the "Times of London." First topic: Afghanistan, loya jirga, great name, the big gathering in Afghanistan, 1,500 delegates to f'

Duplicate #2 | Count: 2
"VELSHI: OK, we've talked about the middle class, we've talked taxes, now we're talking about unemployment benefits. Something you are going to hear a lot about, again, over the next coming months. Why are we continuing to have this discussion about unemployment benefits? Let me give you a few facts. Number one, unemployment benefits have been extended eight times. Federal unemployment benefits, since they were 

In [26]:
from collections import Counter

# Print duplicated full samples within each split
for split in ["train", "validation", "test"]:
    print(f"\n[{split}] duplicated full samples")
    full_sample_list = [
        (example["dialogue"], example["summary"], example["summary_zh"])
        for example in dataset[split]
    ]
    full_sample_counter = Counter(full_sample_list)

    duplicated_items = [(item, count) for item, count in full_sample_counter.items() if count > 1]

    for idx, (item, count) in enumerate(duplicated_items[:3]):
        dialogue, summary, summary_zh = item
        print(f"\nDuplicate #{idx + 1} | Count: {count}")
        print("Dialogue:", repr(dialogue[:300]))
        print("Summary:", repr(summary))
        print("Summary_zh:", repr(summary_zh))


[train] duplicated full samples

Duplicate #1 | Count: 2
Dialogue: 'ROTH: Free speech lives at the U.N., but please, if you\'re going to say something like that, bring a picture or give out a phone number. Well there is a gorgeous woman whose name is not Irina (ph), but Afsane, and she\'s here in the studio with me. Afsane Bassir Pour of the French newspaper "Le Monde'
Summary: 'Can Loya Jirga Help Afghanistan Rebuild?'
Summary_zh: '阿富汗大国民议会能否帮助阿富汗重建？'

Duplicate #2 | Count: 2
Dialogue: "VELSHI: OK, we've talked about the middle class, we've talked taxes, now we're talking about unemployment benefits. Something you are going to hear a lot about, again, over the next coming months. Why are we continuing to have this discussion about unemployment benefits? Let me give you a few facts."
Summary: "Do Jobless Benefits Kill Job Seekers' Drive To Find Work?"
Summary_zh: '失业福利是否扼杀了求职者找工作的动力？'

Duplicate #3 | Count: 2
Dialogue: 'ANDERSON COOPER, CNN HOST: Good evening. Thanks for joining us to

In [29]:
from itertools import combinations

split_dialogues = {
    split: set(example["dialogue"] for example in dataset[split])
    for split in ["train", "validation", "test"]
}

# Check dialogue overlaps across splits
for split_a, split_b in combinations(["train", "validation", "test"], 2):
    overlap = split_dialogues[split_a] & split_dialogues[split_b]
    print(f"{split_a} vs {split_b} dialogue overlap: {len(overlap)}")

train vs validation dialogue overlap: 9
train vs test dialogue overlap: 6
validation vs test dialogue overlap: 2


In [31]:
from itertools import combinations

# Print overlapping full samples across splits
for split_a, split_b in combinations(["train", "validation", "test"], 2):
    overlap = split_full_samples[split_a] & split_full_samples[split_b]
    print(f"\n[{split_a} vs {split_b}] overlap count: {len(overlap)}")

    for idx, item in enumerate(list(overlap)[:3]):
        dialogue, summary, summary_zh = item
        print(f"\nOverlap #{idx + 1}")
        print("Dialogue:", repr(dialogue[:300]))
        print("Summary:", repr(summary))
        print("Summary_zh:", repr(summary_zh))


[train vs validation] overlap count: 9

Overlap #1
Dialogue: 'OSWALD: Paris is certainly world renowned for its cuisine and when it comes to dining out, your choices here are endless. But we wanted to find a way to bring the flavor of Paris back home with us. So we decided to learn to cook like a Parisian and what better place than Le Cordon Bleu?\r\nOSWALD (voi'
Summary: 'The Culinary Universe of Le Cordon Bleu'
Summary_zh: '蓝带厨艺学校的烹饪世界'

Overlap #2
Dialogue: "RICK SANCHEZ, CNN ANCHOR: New York's driver's license issue, this is a controversial plan. Well, Utah has an established plan. In Utah, you can get a driver's license, but it doesn't mean that you're legal. All it means is that you can drive. But here's the question. Does it work? It's actually one "
Summary: 'Farm Worker Shortage Forcing U.S. Companies to Relocate?'
Summary_zh: '农场工人短缺迫使美国公司搬迁？'

Overlap #3
Dialogue: "NGUYEN: It's 16 after the hour. Here are three of the stories that we're working on right here in the CNN NEWS

In [32]:
# Turn Analysis
import re
from collections import Counter
import pandas as pd

# Parse a dialogue into speaker-turn pairs
def parse_dialogue(dialogue_text):
    dialogue_text = dialogue_text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [line.strip() for line in dialogue_text.split("\n") if line.strip()]

    turns = []
    for line in lines:
        # Split only at the first colon
        if ":" in line:
            speaker, utterance = line.split(":", 1)
            speaker = speaker.strip()
            utterance = utterance.strip()
        else:
            # Handle malformed lines without a speaker tag
            speaker = "UNKNOWN"
            utterance = line.strip()

        turns.append({
            "speaker": speaker,
            "utterance": utterance
        })

    return turns


# Compute turn-level features for one dialogue
def compute_turn_level_features(dialogue_text):
    turns = parse_dialogue(dialogue_text)

    num_turns = len(turns)
    speakers = [turn["speaker"] for turn in turns]
    utterances = [turn["utterance"] for turn in turns]

    num_speakers = len(set(speakers))

    turn_char_lengths = [len(u) for u in utterances]
    turn_word_lengths = [len(u.split()) for u in utterances]

    avg_turn_char_len = sum(turn_char_lengths) / num_turns if num_turns > 0 else 0
    avg_turn_word_len = sum(turn_word_lengths) / num_turns if num_turns > 0 else 0

    max_turn_char_len = max(turn_char_lengths) if num_turns > 0 else 0
    max_turn_word_len = max(turn_word_lengths) if num_turns > 0 else 0

    # Ratio of turns spoken by the most frequent speaker
    speaker_counts = Counter(speakers)
    dominant_speaker_turn_ratio = (
        max(speaker_counts.values()) / num_turns if num_turns > 0 else 0
    )

    # Ratio of consecutive turns spoken by the same speaker
    consecutive_same_speaker_count = 0
    consecutive_pair_count = max(num_turns - 1, 0)

    for i in range(1, num_turns):
        if speakers[i] == speakers[i - 1]:
            consecutive_same_speaker_count += 1

    same_speaker_consecutive_ratio = (
        consecutive_same_speaker_count / consecutive_pair_count
        if consecutive_pair_count > 0 else 0
    )

    return {
        "num_turns": num_turns,
        "num_speakers": num_speakers,
        "avg_turn_char_len": avg_turn_char_len,
        "avg_turn_word_len": avg_turn_word_len,
        "max_turn_char_len": max_turn_char_len,
        "max_turn_word_len": max_turn_word_len,
        "dominant_speaker_turn_ratio": dominant_speaker_turn_ratio,
        "same_speaker_consecutive_ratio": same_speaker_consecutive_ratio,
    }


# Add turn-level features to the dataset
def add_turn_features(example):
    features = compute_turn_level_features(example["dialogue"])
    example.update(features)
    return example


dataset_turn = dataset.map(add_turn_features)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [33]:
metrics = [
    "num_turns",
    "num_speakers",
    "avg_turn_char_len",
    "avg_turn_word_len",
    "max_turn_char_len",
    "max_turn_word_len",
    "dominant_speaker_turn_ratio",
    "same_speaker_consecutive_ratio",
]

results = []

for split in ["train", "validation", "test"]:
    for metric in metrics:
        values = dataset_turn[split][metric]

        results.append({
            "split": split,
            "metric": metric,
            "min": round(min(values), 2),
            "max": round(max(values), 2),
            "avg": round(sum(values) / len(values), 2),
        })

df_turn_stats = pd.DataFrame(results)
print(df_turn_stats)

         split                          metric     min      max      avg
0        train                       num_turns    1.00    970.0    30.44
1        train                    num_speakers    1.00    101.0     9.28
2        train               avg_turn_char_len   33.67   9767.5   365.57
3        train               avg_turn_word_len    6.42   1641.5    64.87
4        train               max_turn_char_len   81.00  28703.0  1340.23
5        train               max_turn_word_len   16.00   5380.0   237.04
6        train     dominant_speaker_turn_ratio    0.07      1.0     0.40
7        train  same_speaker_consecutive_ratio    0.00      1.0     0.05
8   validation                       num_turns    1.00    684.0    29.28
9   validation                    num_speakers    1.00     88.0     9.21
10  validation               avg_turn_char_len   47.80   9218.0   367.03
11  validation               avg_turn_word_len    8.90   1672.0    65.17
12  validation               max_turn_char_len  130

In [34]:
for split in ["train", "validation", "test"]:
    print(f"\n[{split}]")
    print("avg num_turns:", round(sum(dataset_turn[split]["num_turns"]) / len(dataset_turn[split]), 2))
    print("avg num_speakers:", round(sum(dataset_turn[split]["num_speakers"]) / len(dataset_turn[split]), 2))
    print("avg avg_turn_char_len:", round(sum(dataset_turn[split]["avg_turn_char_len"]) / len(dataset_turn[split]), 2))
    print("avg avg_turn_word_len:", round(sum(dataset_turn[split]["avg_turn_word_len"]) / len(dataset_turn[split]), 2))
    print("avg max_turn_char_len:", round(sum(dataset_turn[split]["max_turn_char_len"]) / len(dataset_turn[split]), 2))
    print("avg max_turn_word_len:", round(sum(dataset_turn[split]["max_turn_word_len"]) / len(dataset_turn[split]), 2))
    print("avg dominant_speaker_turn_ratio:", round(sum(dataset_turn[split]["dominant_speaker_turn_ratio"]) / len(dataset_turn[split]), 2))
    print("avg same_speaker_consecutive_ratio:", round(sum(dataset_turn[split]["same_speaker_consecutive_ratio"]) / len(dataset_turn[split]), 2))


[train]
avg num_turns: 30.44
avg num_speakers: 9.28
avg avg_turn_char_len: 365.57
avg avg_turn_word_len: 64.87
avg max_turn_char_len: 1340.23
avg max_turn_word_len: 237.04
avg dominant_speaker_turn_ratio: 0.4
avg same_speaker_consecutive_ratio: 0.05

[validation]
avg num_turns: 29.28
avg num_speakers: 9.21
avg avg_turn_char_len: 367.03
avg avg_turn_word_len: 65.17
avg max_turn_char_len: 1337.2
avg max_turn_word_len: 236.47
avg dominant_speaker_turn_ratio: 0.4
avg same_speaker_consecutive_ratio: 0.04

[test]
avg num_turns: 30.56
avg num_speakers: 9.27
avg avg_turn_char_len: 364.78
avg avg_turn_word_len: 64.76
avg max_turn_char_len: 1336.48
avg max_turn_word_len: 236.27
avg dominant_speaker_turn_ratio: 0.4
avg same_speaker_consecutive_ratio: 0.04


In [35]:
example_turns = parse_dialogue(dataset["train"][0]["dialogue"])

for i, turn in enumerate(example_turns[:10]):
    print(f"Turn {i+1}: speaker={turn['speaker']}, utterance={turn['utterance']}")

Turn 1: speaker=MADELEINE BRAND, host, utterance=I'm Madeleine Brand. Chinese Premier Wen Jiabao is in Sichuan province today. He's inspecting the so called quake lake, that's a lake formed by landslides after last month's 7.9 earthquake.
Turn 2: speaker=ALEX CHADWICK, host, utterance=Authorities have evacuated one quarter of a million people in the flood path of that lake. For some it's their second flight to safety, and still many are anxious to get back home. Jamila Trindle reports.
Turn 3: speaker=JAMILA TRINDLE, utterance=Tua Hua Shan or Peach Blossom Mountain is really just a hill, but it's become a refuge for thousands of people from low lying villages nearby. If the dammed river upstream gives way and floods the valley, it should still be above water. For people staying here, it's been three weeks of fear and uncertainty. After living through one disaster and fleeing the threat of another.
Turn 4: speaker=Mr. WEN FONG, utterance=(Through translator)  Everyone wants peace and qu